# Full Workflow Test: Persistent IDs → Timestacks → Labels

End-to-end test of the complete pipeline:
1. Load lake polygons from CSVs (2018 + 2019)
2. Run FeatureTracker per region → persistent IDs
3. Output lookup table
4. Batch-build .nc timestacks (small subset)
5. Launch labeler

Using CW region, 10 lakes only, for speed.

In [1]:
import sys, os
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely import wkt

from sat_tile_stack import sattile_stack, write_netcdf_from_da
from sat_tile_stack.ids import FeatureTracker

print("Imports OK")

Imports OK


## Step 1: Load lake polygons

In [2]:
df_2018 = pd.read_csv("../labeling/labels/labels_2018_volumes.csv")
df_2019 = pd.read_csv("../labeling/labels/labels_2019_volumes.csv")

gdf_2018 = gpd.GeoDataFrame(
    df_2018,
    geometry=df_2018["geometry"].apply(wkt.loads),
    crs="EPSG:4326",
)
gdf_2019 = gpd.GeoDataFrame(
    df_2019,
    geometry=df_2019["geometry"].apply(wkt.loads),
    crs="EPSG:4326",
)

print(f"2018: {len(gdf_2018)} lakes")
print(f"2019: {len(gdf_2019)} lakes")

# Filter to CW region for this test
REGION = "CW"
g18 = gdf_2018[gdf_2018["region"] == REGION].copy().reset_index(drop=True)
g19 = gdf_2019[gdf_2019["region"] == REGION].copy().reset_index(drop=True)
print(f"\nCW 2018: {len(g18)} lakes")
print(f"CW 2019: {len(g19)} lakes")

2018: 3846 lakes
2019: 6146 lakes

CW 2018: 679 lakes
CW 2019: 1000 lakes


## Step 2: Assign persistent IDs

In [3]:
TOLERANCE_M = 500  # meters

tracker = FeatureTracker(method="centroid", tolerance_m=TOLERANCE_M, id_prefix=REGION)
assignments_2018 = tracker.register(g18, "2018")
assignments_2019 = tracker.register(g19, "2019")

tracker.summary()

status = tracker.get_status("2019")
print(f"\n2019 status:")
print(f"  Matched to 2018: {len(status['persistent'])}")
print(f"  New in 2019:     {len(status['new'])}")
print(f"  Missing from 2018: {len(status['missing'])}")

FeatureTracker (method=centroid, tolerance=500)
  Registered 2 time periods: 2018 (679 features), 2019 (1000 features)
  Total persistent features: 1141
  2019: 538 matched, 462 new, 141 missing
  Features in ALL periods: 538

2019 status:
  Matched to 2018: 538
  New in 2019:     462
  Missing from 2018: 141


## Step 3: Build lookup table

In [4]:
# Build a lookup table: persistent_id, region, year, lon, lat, original_id
rows = []

id_table = tracker.get_id_table()
for (year, source_idx), persistent_id in id_table.items():
    if year == "2018":
        gdf = g18
    else:
        gdf = g19
    
    geom = gdf.loc[source_idx, "geometry"]
    original_id = gdf.loc[source_idx, "new_id"] if "new_id" in gdf.columns else f"{REGION}_{source_idx}"
    
    rows.append({
        "persistent_id": persistent_id,
        "region": REGION,
        "year": year,
        "lon": geom.centroid.x,
        "lat": geom.centroid.y,
        "original_id": original_id,
        "nc_filename": f"{persistent_id}_{year}.nc",
    })

lookup_df = pd.DataFrame(rows)
print(f"Lookup table: {len(lookup_df)} entries")
print(f"Unique persistent IDs: {lookup_df['persistent_id'].nunique()}")
print(f"\nSample:")
lookup_df.head(10)

Lookup table: 1679 entries
Unique persistent IDs: 1141

Sample:


,persistent_id,region,year,lon,lat,original_id,nc_filename
0,CW_0000,CW,2018,-50.173528,70.683652,CW2018_1077,CW_0000_2018.nc
1,CW_0001,CW,2018,-49.756356,69.614755,CW2018_1078,CW_0001_2018.nc
2,CW_0002,CW,2018,-49.805573,69.576565,CW2018_1079,CW_0002_2018.nc
3,CW_0003,CW,2018,-49.857634,69.423374,CW2018_1080,CW_0003_2018.nc
4,CW_0004,CW,2018,-49.158831,70.368667,CW2018_1081,CW_0004_2018.nc
5,CW_0005,CW,2018,-49.034856,69.737930,CW2018_1082,CW_0005_2018.nc
6,CW_0006,CW,2018,-48.528049,69.565266,CW2018_1083,CW_0006_2018.nc
7,CW_0007,CW,2018,-47.980637,68.864209,CW2018_1084,CW_0007_2018.nc
8,CW_0008,CW,2018,-48.416684,68.717238,CW2018_1085,CW_0008_2018.nc
9,CW_0009,CW,2018,-49.311003,69.419547,CW2018_1086,CW_0009_2018.nc


In [5]:
# Show some lakes that appear in both years
both = lookup_df.groupby("persistent_id").filter(lambda x: len(x) == 2)
print(f"Lakes in BOTH years: {both['persistent_id'].nunique()}")
print(f"\nExamples:")
for pid in both["persistent_id"].unique()[:5]:
    subset = both[both["persistent_id"] == pid]
    print(f"\n  {pid}:")
    for _, row in subset.iterrows():
        print(f"    {row['year']}: ({row['lon']:.4f}, {row['lat']:.4f}) — {row['original_id']} → {row['nc_filename']}")

Lakes in BOTH years: 538

Examples:

  CW_0000:
    2018: (-50.1735, 70.6837) — CW2018_1077 → CW_0000_2018.nc
    2019: (-50.1802, 70.6818) — CW2019_2356 → CW_0000_2019.nc

  CW_0001:
    2018: (-49.7564, 69.6148) — CW2018_1078 → CW_0001_2018.nc
    2019: (-49.7540, 69.6131) — CW2019_2144 → CW_0001_2019.nc

  CW_0005:
    2018: (-49.0349, 69.7379) — CW2018_1082 → CW_0005_2018.nc
    2019: (-49.0351, 69.7381) — CW2019_2097 → CW_0005_2019.nc

  CW_0006:
    2018: (-48.5280, 69.5653) — CW2018_1083 → CW_0006_2018.nc
    2019: (-48.5280, 69.5654) — CW2019_1949 → CW_0006_2019.nc

  CW_0007:
    2018: (-47.9806, 68.8642) — CW2018_1084 → CW_0007_2018.nc
    2019: (-47.9832, 68.8647) — CW2019_1714 → CW_0007_2019.nc


In [6]:
# Save lookup table
LOOKUP_DIR = Path("../labeling/lookups")
LOOKUP_DIR.mkdir(parents=True, exist_ok=True)
lookup_path = LOOKUP_DIR / f"{REGION}_lookup.csv"
lookup_df.to_csv(lookup_path, index=False)
print(f"Saved lookup table: {lookup_path}")

Saved lookup table: ../labeling/lookups/CW_lookup.csv


## Step 4: Batch-build .nc timestacks (small subset)

Build timestacks for 5 lakes that appear in both 2018 and 2019.
Each lake gets one .nc per year, named `{persistent_id}_{year}.nc`.

In [7]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Pick 5 lakes that appear in both years
both_pids = both["persistent_id"].unique()[:5]
subset = lookup_df[lookup_df["persistent_id"].isin(both_pids)]
print(f"Building stacks for {len(both_pids)} lakes × 2 years = {len(subset)} stacks")
print(f"Lakes: {list(both_pids)}")

Building stacks for 5 lakes × 2 years = 10 stacks
Lakes: ['CW_0000', 'CW_0001', 'CW_0005', 'CW_0006', 'CW_0007']


In [12]:
LABELING_BASE = Path("../labeling")
TIME_RANGES = {"2018": "2018-06-01/2018-06-30", 
               "2019": "2019-06-01/2019-06-30"}
TILE_SIZE = 128
PIX_RES = 20

for _, row in subset.iterrows():
    year = row["year"]
    pid = row["persistent_id"]
    lon, lat = row["lon"], row["lat"]
    
    out_dir = LABELING_BASE / f"{REGION}_{year}" / f"stacks_{REGION}_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)
    outfile = out_dir / row["nc_filename"]
    
    if outfile.exists():
        print(f"  {row["nc_filename"]} exists, skipping")
        continue
    
    print(f"  Building {row["nc_filename"]} at ({lon:.4f}, {lat:.4f})...")
    try:
        stack = sattile_stack(
            catalog, (lon, lat),
            band_names=["B04", "B03", "B02", "B08", "B11", "SCL"],
            collection="sentinel-2-l2a",
            pix_res=PIX_RES, tile_size=TILE_SIZE,
            time_range=TIME_RANGES[year],
            normalize=False,
            cloudmask="scl",
            pull_to_mem=True,
        )
        write_netcdf_from_da(stack, str(outfile))
    except Exception as e:
        print(f"    ERROR: {e}")

# Verify
for year in ["2018", "2019"]:
    d = LABELING_BASE / f"{REGION}_{year}" / "stacks"
    if d.exists():
        nc_files = sorted(d.glob("*.nc"))
        print(f"\n{REGION}_{year}/stacks: {len(nc_files)} stacks")
        for f in nc_files:
            print(f"  {f.name}")


  Building CW_0000_2018.nc at (-50.1735, 70.6837)...
Pulling stack into memory, shape: (30, 7, 128, 128)
[########################################] | 100% Completed | 9.24 ss
Stack loaded, shape: (30, 7, 128, 128)
wrote ../labeling/CW_2018/stacks_CW_2018/CW_0000_2018.nc
  Building CW_0001_2018.nc at (-49.7564, 69.6148)...
Pulling stack into memory, shape: (30, 7, 128, 128)
[########################################] | 100% Completed | 4.61 ss
Stack loaded, shape: (30, 7, 128, 128)
wrote ../labeling/CW_2018/stacks_CW_2018/CW_0001_2018.nc
  Building CW_0005_2018.nc at (-49.0349, 69.7379)...
Pulling stack into memory, shape: (30, 7, 128, 128)
[########################################] | 100% Completed | 7.84 ss
Stack loaded, shape: (30, 7, 128, 128)
wrote ../labeling/CW_2018/stacks_CW_2018/CW_0005_2018.nc
  Building CW_0006_2018.nc at (-48.5280, 69.5653)...
Pulling stack into memory, shape: (30, 7, 128, 128)
[########################################] | 100% Completed | 20.95 s
Stack loaded

## Step 5: Launch the labeler

After `pip install -e ".[labeling]"`, run from any terminal:

    lakelabel --nc_dir labeling/CW_2019/stacks_CW_2019

Labels auto-save to labeling/CW_2019/labels_CW_2019.csv


In [13]:
# Check if any labels exist
for year in ["2018", "2019"]:
    labels_path = Path(f"../labeling/{REGION}_{year}/labels_{REGION}_{year}.csv")
    if labels_path.exists():
        df = pd.read_csv(labels_path)
        print(f"{labels_path.name}: {len(df)} labels")
        print(df[["lake_id", "label"]].to_string(index=False))
        print()
    else:
        print(f"labels_{REGION}_{year}.csv: not yet created")


labels_CW_2018.csv: 5 labels
     lake_id label
CW_0000_2018    ND
CW_0001_2018    LD
CW_0005_2018    ND
CW_0006_2018    ND
CW_0007_2018    ND

labels_CW_2019.csv: 5 labels
     lake_id label
CW_0000_2019    LD
CW_0001_2019    MD
CW_0005_2019    ND
CW_0006_2019    ND
CW_0007_2019    ND



## Step 6: Cross-year analysis

After labeling both years, compare labels for the same lake across years.

In [14]:
# Load and merge labels from both years
all_labels = []
for year in ["2018", "2019"]:
    labels_path = Path(f"../labeling/{REGION}_{year}/labels_{REGION}_{year}.csv")
    if labels_path.exists():
        df = pd.read_csv(labels_path)
        df["persistent_id"] = df["lake_id"].apply(lambda x: x.rsplit("_", 1)[0])
        df["year"] = year
        all_labels.append(df)

if all_labels:
    merged = pd.concat(all_labels, ignore_index=True)
    print(f"Total labels: {len(merged)}")
    
    both_labeled = merged.groupby("persistent_id").filter(lambda x: len(x) == 2)
    if len(both_labeled) > 0:
        print(f"\nLakes labeled in both years: {both_labeled["persistent_id"].nunique()}")
        print("\nCross-year comparison:")
        for pid in both_labeled["persistent_id"].unique():
            rows = both_labeled[both_labeled["persistent_id"] == pid]
            labels = dict(zip(rows["year"], rows["label"]))
            match = "✓" if labels.get("2018") == labels.get("2019") else "✗"
            print(f"  {pid}: 2018={labels.get("2018", "?")}, 2019={labels.get("2019", "?")} {match}")
    else:
        print("No lakes labeled in both years yet.")
else:
    print("No labels found. Run the labeler first!")


Total labels: 10

Lakes labeled in both years: 5

Cross-year comparison:
  CW_0000: 2018=ND, 2019=LD ✗
  CW_0001: 2018=LD, 2019=MD ✗
  CW_0005: 2018=ND, 2019=ND ✓
  CW_0006: 2018=ND, 2019=ND ✓
  CW_0007: 2018=ND, 2019=ND ✓
